# 提示词 — System Prompt 与 Dynamic Prompt

System Prompt 是控制 Agent 行为的核心手段。

In [ ]:
from dotenv import load_dotenv
load_dotenv()

print("环境变量已加载")

## system_prompt 参数

接受字符串或 SystemMessage 对象。

In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import SystemMessage

model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)
agent = create_agent(model=model, system_prompt="你是菜鸟教程 RUNOOB 的学习顾问。")
system_msg = SystemMessage(content="你是菜鸟教程 RUNOOB 的学习顾问。")
agent = create_agent(model=model, system_prompt=system_msg)
print("两种方式等价")

## 设计有效的 System Prompt

应包含：角色定义、行为准则、工具使用指引、边界约束。

In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langchain.tools import tool

@tool
def search_course(keyword: str) -> str:
    courses = {"python": "Python3 基础教程（免费）", "html": "HTML 基础教程（免费）"}
    return courses.get(keyword.lower(), "未找到")

GOOD_PROMPT = """你是菜鸟教程 RUNOOB 的学习顾问。
回答简洁，不超过 3 句话。
优先使用 search_course 工具查询课程。"""

model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)
agent = create_agent(model=model, tools=[search_course], system_prompt=GOOD_PROMPT)
print("带 system_prompt 的 Agent 已创建")

## @dynamic_prompt——动态生成提示词

在每次模型调用前动态生成 system_prompt。

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt
from langchain.agents.middleware.types import ModelRequest
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langchain.tools import tool

@tool
def search_course(keyword: str) -> str:
    courses = {"python": "Python3 基础教程（免费）", "html": "HTML 基础教程（免费）"}
    return courses.get(keyword.lower(), "未找到")

@dynamic_prompt
def personalized_prompt(request: ModelRequest) -> str:
    messages = request.state.get("messages", [])
    base = "你是菜鸟教程 RUNOOB 的学习顾问。"
    if len(messages) <= 2:
        return base + "用户刚开始对话，请热情问候。"
    return base + "回答尽量简洁。"

model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)
agent = create_agent(model=model, tools=[search_course], middleware=[personalized_prompt])
print("动态提示词 Agent 已创建")

### 进阶——结合运行时上下文

`request` 提供 state 和 runtime.context。

In [ ]:
from datetime import datetime
from langchain.agents.middleware import dynamic_prompt
from langchain.agents.middleware.types import ModelRequest

@dynamic_prompt
def context_aware_prompt(request: ModelRequest) -> str:
    context = request.runtime.context or {}
    user_name = context.get("user_name", "同学")
    now = datetime.now()
    greeting = "早上好" if now.hour < 12 else "下午好"
    return f"你是菜鸟教程的学习顾问。\n称呼用户为{user_name}。\n{greeting}！"

print("上下文感知动态提示词已定义")

## System Prompt 优先级

@dynamic_prompt middleware 优先级高于 create_agent 的 system_prompt。

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt
from langchain.chat_models import init_chat_model

@dynamic_prompt
def override_prompt(request) -> str:
    return "你是一只猫，所有回复用'喵'结尾。"

agent = create_agent(model=init_chat_model("deepseek:deepseek-v4-flash", temperature=0.7), system_prompt="你是专业顾问。", middleware=[override_prompt])
print("middleware 会覆盖静态 system_prompt")

## System Prompt 设计清单

| 要素 | 说明 |
| --- | --- |
| 角色定义 | 明确 AI 身份 |
| 行为准则 | 约束回复风格 |
| 工具指引 | 何时使用工具 |
| 边界约束 | 什么不能做 |

**术语：** System Prompt、@dynamic_prompt、ModelRequest